# Part 2 — Fine-Tuning, Evaluation & All Tests

**Thesis:** *Streaming NIDS via Self-Supervised BiMamba with TED and Blockwise Early Exit*

This notebook covers the **second half** — all experiments, tests, and visualizations:
1. Fine-tune 5 teacher models (BiMamba×3, BERT×2)
2. ML baselines (XGBoost, Random Forest)
3. Train 4 student variants (No-KD, Standard-KD, Uniform-KD, TED)
4. TED weight ablation study
5. Dynamic early exit evaluation (threshold sweep)
6. Cross-dataset generalization (UNSW → CIC-IDS, CTU-13)
7. Per-attack F1 breakdown (Binary vs Macro F1)
8. Time-to-Detection from raw packet data
9. Latency & throughput benchmarking
10. Multi-seed stability analysis
11. Few-shot adaptation with SSL
12. Comprehensive visualizations (t-SNE, Pareto, radar, histograms)

All weights are saved to `weights/teachers/` and `weights/students/`.

## 2.1 Imports

In [2]:
import sys, os, time, pickle, warnings, copy, json, math, traceback
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from collections import Counter, OrderedDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, recall_score, precision_score,
                             classification_report)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
warnings.filterwarnings('ignore')
%matplotlib inline
print("All imports successful.")

All imports successful.


## 2.2 Path Configuration & Device

In [3]:
WORKSPACE  = "/home/T2510596/Downloads/totally fresh"
ORG_FINAL  = os.path.join(WORKSPACE, "Organized_Final")
THESIS_DIR = os.path.join(WORKSPACE, "thesis_final")
DATA_DIR   = os.path.join(ORG_FINAL, "data", "unswnb15_full")

PRETRAIN_PKL  = os.path.join(DATA_DIR, "pretrain_50pct_benign.pkl")
FINETUNE_PKL  = os.path.join(DATA_DIR, "finetune_mixed.pkl")
FLOWS_ALL_PKL = os.path.join(DATA_DIR, "flows_all.pkl")

CIC_PKL = os.path.join(THESIS_DIR, "data", "cicdos2019_flows.pkl")
CTU_PKL = os.path.join(THESIS_DIR, "data", "ctu13_flows.pkl")

EXISTING_SSL_MASKING = os.path.join(ORG_FINAL, "Final_Submission", "checkpoints", "encoder_masking.pth")
EXISTING_SSL_CUTMIX  = os.path.join(ORG_FINAL, "Final_Submission", "checkpoints", "encoder_cutmix.pth")

WEIGHT_DIR  = os.path.join(THESIS_DIR, "weights")
SSL_DIR     = os.path.join(WEIGHT_DIR, "ssl")
TEACHER_DIR = os.path.join(WEIGHT_DIR, "teachers")
STUDENT_DIR = os.path.join(WEIGHT_DIR, "students")
CKPT_DIR    = os.path.join(THESIS_DIR, "checkpoints")
PLOT_DIR    = os.path.join(THESIS_DIR, "plots")
RESULT_DIR  = os.path.join(THESIS_DIR, "results")

for d in [SSL_DIR, TEACHER_DIR, STUDENT_DIR, PLOT_DIR, RESULT_DIR]:
    os.makedirs(d, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")

Device: cuda
GPU: NVIDIA GeForce RTX 4070 Ti SUPER, VRAM: 16.7GB


## 2.3 Model Definitions

All architecture classes needed for this notebook (same as Part 1 + Student models).

In [4]:
# ── Mamba SSM ──
class Mamba(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_inner = int(expand * d_model)
        self.dt_rank = math.ceil(d_model / 16)
        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(self.d_inner, self.d_inner, bias=True,
                                kernel_size=d_conv, groups=self.d_inner, padding=d_conv - 1)
        self.x_proj = nn.Linear(self.d_inner, self.dt_rank + d_state * 2, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
        A = torch.arange(1, d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(self.d_inner))

    def forward(self, x):
        B, L, _ = x.shape
        xz = self.in_proj(x)
        x_inner, z = xz.chunk(2, dim=-1)
        x_c = self.conv1d(x_inner.permute(0,2,1))[:,:,:L].permute(0,2,1)
        x_c = F.silu(x_c)
        x_dbl = self.x_proj(x_c)
        dt, B_s, C_s = torch.split(x_dbl, [self.dt_rank, self.d_state, self.d_state], dim=-1)
        dt = F.softplus(self.dt_proj(dt))
        A = -torch.exp(self.A_log.float())
        ys = []
        h = torch.zeros(B, self.d_inner, self.d_state, device=x.device)
        for t in range(L):
            dt_t = dt[:,t,:].unsqueeze(-1)
            h = torch.exp(A * dt_t) * h + (dt_t * B_s[:,t,:].unsqueeze(1)) * x_c[:,t,:].unsqueeze(-1)
            ys.append(torch.matmul(h, C_s[:,t,:].unsqueeze(-1)).squeeze(-1))
        y = torch.stack(ys, dim=1) + x_c * self.D
        return self.out_proj(y * F.silu(z))

# ── PacketEmbedder ──
class PacketEmbedder(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.emb_proto = nn.Embedding(256, 32)
        self.emb_flags = nn.Embedding(64, 32)
        self.emb_dir = nn.Embedding(2, 8)
        self.proj_len = nn.Linear(1, 32)
        self.proj_iat = nn.Linear(1, 32)
        self.fusion = nn.Linear(136, d_model)
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x):
        proto = x[:,:,0].long().clamp(0,255)
        length = x[:,:,1].unsqueeze(-1)
        flags = x[:,:,2].long().clamp(0,63)
        iat = x[:,:,3].unsqueeze(-1)
        direction = x[:,:,4].long().clamp(0,1)
        cat = torch.cat([self.emb_proto(proto), self.emb_flags(flags),
                         self.emb_dir(direction), self.proj_len(length),
                         self.proj_iat(iat)], dim=-1)
        return self.norm(self.fusion(cat))

# ── BiMambaEncoder ──
class BiMambaEncoder(nn.Module):
    def __init__(self, d_model=256, n_layers=4):
        super().__init__()
        self.tokenizer = PacketEmbedder(d_model)
        self.layers = nn.ModuleList([Mamba(d_model, 16, 4, 2) for _ in range(n_layers)])
        self.layers_rev = nn.ModuleList([Mamba(d_model, 16, 4, 2) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.proj_head = nn.Sequential(nn.Linear(d_model, d_model), nn.ReLU(), nn.Linear(d_model, 128))
        self.recon_head = nn.Linear(d_model, 5)
    def forward(self, x):
        feat = self.tokenizer(x)
        for fwd, bwd in zip(self.layers, self.layers_rev):
            feat = self.norm(fwd(feat) + torch.flip(bwd(torch.flip(feat, [1])), [1]) + feat)
        g = feat.mean(1)
        return self.proj_head(g), self.recon_head(feat)

# ── BertEncoder ──
class LearnablePositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        self.pe_emb = nn.Embedding(max_len, d_model)
    def forward(self, x):
        return x + self.pe_emb(torch.arange(x.size(1), device=x.device).unsqueeze(0))

class BertEncoder(nn.Module):
    def __init__(self, d_model=256, n_layers=4, n_heads=8):
        super().__init__()
        self.tokenizer = PacketEmbedder(d_model)
        self.pos_encoder = LearnablePositionalEncoding(d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model, n_heads, d_model*4, batch_first=True, dropout=0.1)
        self.transformer_encoder = nn.TransformerEncoder(enc_layer, n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.proj_head = nn.Sequential(nn.Linear(d_model, d_model), nn.ReLU(), nn.Linear(d_model, 128))
        self.recon_head = nn.Linear(d_model, 5)
    def forward(self, x):
        feat = self.norm(self.transformer_encoder(self.pos_encoder(self.tokenizer(x))))
        g = feat.mean(1)
        return self.proj_head(g), self.recon_head(feat)

# ── UniMambaEncoder (Student backbone — forward only) ──
class UniMambaEncoder(nn.Module):
    def __init__(self, d_model=256, n_layers=4):
        super().__init__()
        self.tokenizer = PacketEmbedder(d_model)
        self.layers = nn.ModuleList([Mamba(d_model, 16, 4, 2) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.d_model = d_model
    def forward(self, x, return_hidden=False):
        feat = self.tokenizer(x)
        hiddens = []
        for layer in self.layers:
            feat = self.norm(layer(feat) + feat)
            if return_hidden: hiddens.append(feat)
        g = feat.mean(1)
        return (g, feat, hiddens) if return_hidden else (g, feat)

# ── Classifier head ──
class Classifier(nn.Module):
    def __init__(self, encoder, use_proj=True):
        super().__init__()
        self.encoder = encoder
        self.use_proj = use_proj
        if use_proj and hasattr(encoder, 'proj_head'):
            self.head = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 2))
        else:
            self.head = nn.Sequential(nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 2))
    def forward(self, x):
        if self.use_proj and hasattr(self.encoder, 'proj_head'):
            z, _ = self.encoder(x)
            return self.head(z)
        else:
            if isinstance(self.encoder, UniMambaEncoder):
                g, _ = self.encoder(x)
            else:
                g, _ = self.encoder(x)
            return self.head(g)

# ── BlockwiseStudent — exits at {8, 16, 32} packets ──
class BlockwiseStudent(nn.Module):
    def __init__(self, d_model=256, n_layers=4, exit_points=[8, 16, 32]):
        super().__init__()
        self.d_model = d_model
        self.exit_points = exit_points
        self.tokenizer = PacketEmbedder(d_model)
        self.layers = nn.ModuleList([Mamba(d_model, 16, 4, 2) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.exit_classifiers = nn.ModuleDict({
            str(ep): nn.Sequential(nn.Linear(d_model, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 2))
            for ep in exit_points})
        self.confidence_heads = nn.ModuleDict({
            str(ep): nn.Sequential(nn.Linear(d_model + 2, 64), nn.ReLU(), nn.Linear(64, 1), nn.Sigmoid())
            for ep in exit_points})

    def _backbone(self, x_emb):
        feat = x_emb
        for layer in self.layers:
            feat = self.norm(layer(feat) + feat)
        return feat

    def forward_at_exit(self, x, ep):
        feat = self._backbone(self.tokenizer(x[:, :ep, :]))
        pooled = feat.mean(1)
        return self.exit_classifiers[str(ep)](pooled), pooled

    def forward_all_exits(self, x):
        feat = self._backbone(self.tokenizer(x))
        results = {}
        for ep in self.exit_points:
            pooled = feat[:, :ep, :].mean(1)
            logits = self.exit_classifiers[str(ep)](pooled)
            conf = self.confidence_heads[str(ep)](torch.cat([pooled, logits], 1)).squeeze(-1)
            results[ep] = {'logits': logits, 'pooled': pooled, 'confidence': conf}
        return results

    def forward_dynamic(self, x, threshold=0.9):
        results = self.forward_all_exits(x)
        B = x.size(0)
        final_logits = torch.zeros(B, 2, device=x.device)
        exit_used = torch.full((B,), self.exit_points[-1], dtype=torch.long, device=x.device)
        exited = torch.zeros(B, dtype=torch.bool, device=x.device)
        for ep in self.exit_points:
            if exited.all(): break
            should = (~exited) & ((results[ep]['confidence'] >= threshold) | (ep == self.exit_points[-1]))
            final_logits[should] = results[ep]['logits'][should]
            exit_used[should] = ep
            exited |= should
        return final_logits, exit_used

print("All model classes defined.")
_s = BlockwiseStudent()
print(f"BlockwiseStudent: {sum(p.numel() for p in _s.parameters()):,} params")
del _s

All model classes defined.
BlockwiseStudent: 1,946,905 params


## 2.4 Load & Split Data

Split the 834K mixed flows into:
- **Train pool** (70%) → take 10% fewshot → split into train (80%) + val (20%)
- **Test** (30%) — held out, never seen during training

In [5]:
class FlowDataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        row = self.data[idx]
        f = row['features'] if isinstance(row, dict) else row
        l = row['label'] if isinstance(row, dict) else 0
        return torch.from_numpy(f).float(), torch.tensor(l).long()

def load_data(fewshot_pct=0.10):
    print("Loading finetune_mixed.pkl...")
    with open(FINETUNE_PKL, 'rb') as f:
        data = pickle.load(f)
    labels = np.array([d['label'] for d in data])
    print(f"  Total: {len(data):,} (benign={sum(labels==0):,}, attack={sum(labels==1):,})")

    train_idx, test_idx = train_test_split(
        np.arange(len(data)), test_size=0.3, stratify=labels, random_state=42)
    train_labels = labels[train_idx]
    target_n = max(int(len(data) * fewshot_pct), 100)
    if target_n >= len(train_idx):
        fs_idx = train_idx
    else:
        fs_idx, _ = train_test_split(train_idx, train_size=target_n,
                                     stratify=train_labels, random_state=42)
    fs_labels = labels[fs_idx]
    tr_idx, val_idx = train_test_split(fs_idx, test_size=0.2,
                                        stratify=fs_labels, random_state=42)
    train_data = [data[i] for i in tr_idx]
    val_data = [data[i] for i in val_idx]
    test_data = [data[i] for i in test_idx]
    print(f"  Train: {len(train_data):,}  Val: {len(val_data):,}  Test: {len(test_data):,}")
    return train_data, val_data, test_data, data

train_data, val_data, test_data, all_data = load_data(fewshot_pct=0.10)
test_loader = DataLoader(FlowDataset(test_data), batch_size=64, num_workers=2)

Loading finetune_mixed.pkl...
  Total: 834,241 (benign=787,005, attack=47,236)
  Train: 66,739  Val: 16,685  Test: 250,273


## 2.5 Training & Evaluation Utilities

Common functions used for all model training.

In [6]:
def get_class_weights(data):
    labels = [d['label'] for d in data]
    c = Counter(labels)
    n = len(labels)
    return torch.tensor([n/(2*max(c[0],1)), n/(2*max(c[1],1))], dtype=torch.float32).to(DEVICE)

def evaluate_model(model, loader, return_preds=False):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            all_probs.extend(torch.softmax(logits, 1)[:, 1].cpu().numpy())
    all_preds, all_labels, all_probs = np.array(all_preds), np.array(all_labels), np.array(all_probs)
    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds, labels=[0,1]).ravel()
    metrics = {
        'acc': accuracy_score(all_labels, all_preds),
        'f1': f1_score(all_labels, all_preds, zero_division=0),
        'auc': roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.5,
        'precision': precision_score(all_labels, all_preds, zero_division=0),
        'recall': recall_score(all_labels, all_preds, zero_division=0),
        'fpr': fp / (fp + tn) if (fp + tn) > 0 else 0,
        'tp': int(tp), 'fp': int(fp), 'tn': int(tn), 'fn': int(fn)
    }
    return (metrics, all_preds, all_labels, all_probs) if return_preds else metrics

def train_classifier(model, train_data, val_data, epochs=30, lr=1e-4,
                      batch_size=32, patience=5, name="model", save_path=None):
    train_loader = DataLoader(FlowDataset(train_data), batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(FlowDataset(val_data), batch_size=batch_size, num_workers=2, pin_memory=True)
    class_w = get_class_weights(train_data)
    criterion = nn.CrossEntropyLoss(weight=class_w)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)
    best_f1 = -1
    best_wts = copy.deepcopy(model.state_dict())
    pat = 0
    model.to(DEVICE)
    for epoch in range(epochs):
        model.train()
        tloss, corr, tot = 0, 0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tloss += loss.item() * x.size(0)
            corr += (logits.argmax(1) == y).sum().item()
            tot += x.size(0)
        vm = evaluate_model(model, val_loader)
        scheduler.step(1 - vm['f1'])
        print(f"  [{name}] E{epoch+1:>2}/{epochs} TrL={tloss/tot:.4f} TrA={corr/tot:.3f} VF1={vm['f1']:.4f} VAUC={vm['auc']:.4f}")
        if vm['f1'] > best_f1:
            best_f1 = vm['f1']; best_wts = copy.deepcopy(model.state_dict()); pat = 0
        else:
            pat += 1
            if pat >= patience:
                print(f"  [{name}] Early stopping at epoch {epoch+1}"); break
    model.load_state_dict(best_wts)
    if save_path:
        torch.save(model.state_dict(), save_path)
        print(f"  [{name}] Saved: {save_path}")
    return model, best_f1

print("Training utilities defined.")

Training utilities defined.


## 2.6 Distillation Loss Functions (Contribution 3 — TED)

Three KD strategies compared:
- **Standard KD**: Distill only at packet 32 (full sequence)
- **Uniform KD**: Equal weight at all exit points {8, 16, 32}
- **TED (Ours)**: Temporally-Emphasised — weight 2:1:0.5 for exits {8:16:32}
  → Forces student to learn early detection signal from teacher

In [7]:
def ted_loss(student_logits, teacher_logits, labels, class_w,
             exit_weights={8: 2.0, 16: 1.0, 32: 0.5}, T=4.0, alpha=0.5):
    total, tw = 0, sum(exit_weights.values())
    for ep, w in exit_weights.items():
        s = student_logits[ep]
        kd = F.kl_div(F.log_softmax(s/T, 1), F.softmax(teacher_logits/T, 1), reduction='batchmean') * T*T
        ce = F.cross_entropy(s, labels, weight=class_w)
        total += (w / tw) * (alpha * kd + (1-alpha) * ce)
    return total

def standard_kd_loss(s32, teacher_logits, labels, class_w, T=4.0, alpha=0.5):
    kd = F.kl_div(F.log_softmax(s32/T, 1), F.softmax(teacher_logits/T, 1), reduction='batchmean') * T*T
    return alpha * kd + (1-alpha) * F.cross_entropy(s32, labels, weight=class_w)

def uniform_kd_loss(student_logits, teacher_logits, labels, class_w, eps=[8,16,32], T=4.0, alpha=0.5):
    total = 0
    for ep in eps:
        s = student_logits[ep]
        kd = F.kl_div(F.log_softmax(s/T, 1), F.softmax(teacher_logits/T, 1), reduction='batchmean') * T*T
        total += alpha * kd + (1-alpha) * F.cross_entropy(s, labels, weight=class_w)
    return total / len(eps)

def train_student(student, teacher, train_data, val_data, kd_mode="ted",
                  epochs=30, lr=1e-3, batch_size=32, patience=5, name="student", save_path=None):
    train_loader = DataLoader(FlowDataset(train_data), batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(FlowDataset(val_data), batch_size=batch_size, num_workers=2, pin_memory=True)
    class_w = get_class_weights(train_data)
    criterion = nn.CrossEntropyLoss(weight=class_w)
    optimizer = torch.optim.AdamW(student.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)
    student.to(DEVICE)
    if teacher is not None:
        teacher.to(DEVICE); teacher.eval()
        for p in teacher.parameters(): p.requires_grad = False
    best_f1, best_wts, pat = -1, copy.deepcopy(student.state_dict()), 0
    for epoch in range(epochs):
        student.train(); tloss, tot = 0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            results = student.forward_all_exits(x)
            if kd_mode == "none":
                loss = sum(criterion(results[ep]['logits'], y) for ep in student.exit_points) / len(student.exit_points)
            elif kd_mode == "standard":
                with torch.no_grad(): t_logits = teacher(x)
                loss = standard_kd_loss(results[32]['logits'], t_logits, y, class_w)
                for ep in [8, 16]: loss += 0.1 * criterion(results[ep]['logits'], y)
            elif kd_mode == "uniform":
                with torch.no_grad(): t_logits = teacher(x)
                loss = uniform_kd_loss({ep: results[ep]['logits'] for ep in student.exit_points}, t_logits, y, class_w)
            elif kd_mode == "ted":
                with torch.no_grad(): t_logits = teacher(x)
                loss = ted_loss({ep: results[ep]['logits'] for ep in student.exit_points}, t_logits, y, class_w)
            # Confidence head loss
            for ep in student.exit_points:
                correct = (results[ep]['logits'].argmax(1) == y).float()
                loss += 0.1 * F.binary_cross_entropy(results[ep]['confidence'], correct)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            optimizer.step()
            tloss += loss.item() * x.size(0); tot += x.size(0)
        # Validate at pkt 32
        student.eval()
        vp, vl = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                vp.extend(student.forward_all_exits(x)[32]['logits'].argmax(1).cpu().numpy())
                vl.extend(y.cpu().numpy())
        vf1 = f1_score(vl, vp, zero_division=0)
        scheduler.step(1 - vf1)
        print(f"  [{name}] E{epoch+1:>2}/{epochs} TrL={tloss/tot:.4f} VF1={vf1:.4f}")
        if vf1 > best_f1:
            best_f1 = vf1; best_wts = copy.deepcopy(student.state_dict()); pat = 0
        else:
            pat += 1
            if pat >= patience: print(f"  [{name}] Early stopping at epoch {epoch+1}"); break
    student.load_state_dict(best_wts)
    if save_path: torch.save(student.state_dict(), save_path); print(f"  [{name}] Saved: {save_path}")
    if teacher is not None: teacher.cpu()
    return student, best_f1

print("KD loss functions and student training defined.")

KD loss functions and student training defined.


## 2.7 Fine-tune 5 Teacher Models

| # | Model | SSL | Purpose |
|---|-------|-----|---------|
| 1 | BiMamba + Masking SSL | ✅ Anti-Shortcut | Best teacher (C1+C2) |
| 2 | BiMamba + CutMix SSL | ✅ CutMix | SSL strategy comparison |
| 3 | BiMamba Scratch | ❌ | No-SSL baseline |
| 4 | BERT + Masking SSL | ✅ Anti-Shortcut | Architecture comparison |
| 5 | BERT Scratch | ❌ | Full baseline |

In [8]:
teacher_configs = OrderedDict({
    "bimamba_masking":  ("BiMamba+MaskSSL",  BiMambaEncoder, os.path.join(SSL_DIR, "bimamba_masking_ssl.pth")),
    "bimamba_cutmix":   ("BiMamba+CutMixSSL", BiMambaEncoder, EXISTING_SSL_CUTMIX),
    "bimamba_scratch":  ("BiMamba_Scratch",    BiMambaEncoder, None),
    "bert_masking":     ("BERT+MaskSSL",       BertEncoder,    os.path.join(SSL_DIR, "bert_masking_ssl.pth")),
    "bert_scratch":     ("BERT_Scratch",        BertEncoder,    None),
})

teacher_results = {}
teacher_models = {}

for key, (display_name, enc_cls, ssl_path) in teacher_configs.items():
    save_path = os.path.join(TEACHER_DIR, f"teacher_{key}.pth")
    # Also check old checkpoint dir
    old_path = os.path.join(CKPT_DIR, f"teacher_{key}.pth")
    if not os.path.exists(save_path) and os.path.exists(old_path):
        import shutil; shutil.copy2(old_path, save_path)

    encoder = enc_cls(d_model=256)
    if ssl_path and os.path.exists(ssl_path):
        encoder.load_state_dict(torch.load(ssl_path, map_location='cpu', weights_only=False))
        print(f"  Loaded SSL: {ssl_path}")

    model = Classifier(encoder, use_proj=True)

    if os.path.exists(save_path):
        print(f"[CACHED] {display_name}")
        model.load_state_dict(torch.load(save_path, map_location='cpu', weights_only=False))
        model.to(DEVICE)
    else:
        print(f"\nTraining: {display_name}")
        model, _ = train_classifier(model, train_data, val_data, epochs=30, lr=1e-4,
                                     batch_size=32, patience=5, name=display_name, save_path=save_path)

    metrics = evaluate_model(model, test_loader)
    teacher_results[key] = metrics
    teacher_models[key] = model
    print(f"  {display_name}: F1={metrics['f1']:.4f} AUC={metrics['auc']:.4f} Recall={metrics['recall']:.4f}")
    model.cpu(); torch.cuda.empty_cache()

with open(os.path.join(RESULT_DIR, 'teacher_results_nb.json'), 'w') as f:
    json.dump(teacher_results, f, indent=2)
print("\nTeacher results saved.")

  Loaded SSL: /home/T2510596/Downloads/totally fresh/thesis_final/weights/ssl/bimamba_masking_ssl.pth
[CACHED] BiMamba+MaskSSL
  BiMamba+MaskSSL: F1=0.8800 AUC=0.9949 Recall=0.9928
  Loaded SSL: /home/T2510596/Downloads/totally fresh/Organized_Final/Final_Submission/checkpoints/encoder_cutmix.pth
[CACHED] BiMamba+CutMixSSL
  BiMamba+CutMixSSL: F1=0.8810 AUC=0.9965 Recall=0.9935
[CACHED] BiMamba_Scratch
  BiMamba_Scratch: F1=0.8847 AUC=0.9957 Recall=0.9925
  Loaded SSL: /home/T2510596/Downloads/totally fresh/thesis_final/weights/ssl/bert_masking_ssl.pth
[CACHED] BERT+MaskSSL
  BERT+MaskSSL: F1=0.0000 AUC=0.3690 Recall=0.0000
[CACHED] BERT_Scratch
  BERT_Scratch: F1=0.8807 AUC=0.9943 Recall=0.9950

Teacher results saved.


## 2.7b — Full-Data Train + Test (No Few-Shot, No Split)

Train every teacher on **100% of the 834K dataset** (90/10 internal split for early stopping only),  
then test on **all 834K** — gives the upper-bound metrics for each architecture.

In [9]:
# ════════════════════════════════════════════════════════════
# FULL TRAIN-TEST: Train on ALL data, Test on ALL data
# ════════════════════════════════════════════════════════════
print("=" * 60)
print("FULL TRAIN-TEST  (No few-shot, all 834K samples)")
print("=" * 60)

# ── Prepare full dataset ──────────────────────────────────
full_data = all_data  # all 834K
full_labels = np.array([d['label'] for d in full_data])
print(f"Full dataset: {len(full_data):,}  "
      f"(benign={sum(full_labels==0):,}, attack={sum(full_labels==1):,})")

# Internal 90/10 split for early-stopping ONLY
full_train_idx, full_val_idx = train_test_split(
    np.arange(len(full_data)), test_size=0.10,
    stratify=full_labels, random_state=42
)
full_train_data = [full_data[i] for i in full_train_idx]
full_val_data   = [full_data[i] for i in full_val_idx]
full_test_loader = DataLoader(FlowDataset(full_data),
                              batch_size=128, num_workers=2)

print(f"  Train: {len(full_train_data):,}   Val: {len(full_val_data):,}   "
      f"Test (=ALL): {len(full_data):,}\n")

# ── Train & Evaluate all 5 teachers on full data ─────────
full_results = {}

for key, (display_name, enc_cls, ssl_path) in teacher_configs.items():
    save_path = os.path.join(TEACHER_DIR, f"teacher_{key}_fulldata.pth")

    encoder = enc_cls(d_model=256)
    if ssl_path and os.path.exists(ssl_path):
        encoder.load_state_dict(
            torch.load(ssl_path, map_location='cpu', weights_only=False))

    model = Classifier(encoder, use_proj=True)

    if os.path.exists(save_path):
        print(f"[CACHED] {display_name} (full-data)")
        model.load_state_dict(
            torch.load(save_path, map_location='cpu', weights_only=False))
        model.to(DEVICE)
    else:
        print(f"\nTraining: {display_name}  (full-data, {len(full_train_data):,} samples)")
        model, _ = train_classifier(
            model, full_train_data, full_val_data,
            epochs=20, lr=1e-4, batch_size=128,
            patience=5, name=f"{display_name}_FULL",
            save_path=save_path)

    metrics = evaluate_model(model, full_test_loader)
    full_results[key] = metrics
    print(f"  {display_name} FULL ▸ F1={metrics['f1']:.4f}  "
          f"AUC={metrics['auc']:.4f}  Recall={metrics['recall']:.4f}  "
          f"Prec={metrics['precision']:.4f}")
    model.cpu(); torch.cuda.empty_cache()

# ── Comparison table: 30% test  vs  Full data ────────────
print("\n" + "=" * 90)
print(f"{'Model':<25} │ {'F1 (30% test)':>13} │ {'F1 (Full)':>10} │ "
      f"{'AUC (30%)':>10} │ {'AUC (Full)':>10}")
print("─" * 90)
for key in teacher_configs:
    old = teacher_results.get(key, {})
    new = full_results[key]
    name = teacher_configs[key][0]
    print(f"{name:<25} │ {old.get('f1',0):>13.4f} │ {new['f1']:>10.4f} │ "
          f"{old.get('auc',0):>10.4f} │ {new['auc']:>10.4f}")
print("=" * 90)

# Save
with open(os.path.join(RESULT_DIR, 'teacher_results_fulldata.json'), 'w') as f:
    json.dump(full_results, f, indent=2)
print("\n✓ Full train-test results saved.")

FULL TRAIN-TEST  (No few-shot, all 834K samples)
Full dataset: 834,241  (benign=787,005, attack=47,236)
  Train: 750,816   Val: 83,425   Test (=ALL): 834,241


Training: BiMamba+MaskSSL  (full-data, 750,816 samples)
  [BiMamba+MaskSSL_FULL] E 1/20 TrL=0.0676 TrA=0.978 VF1=0.8821 VAUC=0.9948
  [BiMamba+MaskSSL_FULL] E 2/20 TrL=0.0482 TrA=0.984 VF1=0.8818 VAUC=0.9949
  [BiMamba+MaskSSL_FULL] E 3/20 TrL=0.0462 TrA=0.985 VF1=0.8832 VAUC=0.9957
  [BiMamba+MaskSSL_FULL] E 4/20 TrL=0.0446 TrA=0.985 VF1=0.8822 VAUC=0.9959
  [BiMamba+MaskSSL_FULL] E 5/20 TrL=0.0437 TrA=0.985 VF1=0.8846 VAUC=0.9964
  [BiMamba+MaskSSL_FULL] E 6/20 TrL=0.0425 TrA=0.985 VF1=0.8797 VAUC=0.9966
  [BiMamba+MaskSSL_FULL] E 7/20 TrL=0.0411 TrA=0.985 VF1=0.8841 VAUC=0.9968
  [BiMamba+MaskSSL_FULL] E 8/20 TrL=0.0403 TrA=0.985 VF1=0.8845 VAUC=0.9966
  [BiMamba+MaskSSL_FULL] E 9/20 TrL=0.0385 TrA=0.985 VF1=0.8843 VAUC=0.9971
  [BiMamba+MaskSSL_FULL] E10/20 TrL=0.0373 TrA=0.985 VF1=0.8840 VAUC=0.9971
  [BiMamba+MaskSSL_FULL]

KeyboardInterrupt: 

## 2.8 Teacher Comparison — Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
names = list(teacher_results.keys())
display = [n.replace('_', '\n') for n in names]
f1s = [teacher_results[n]['f1'] for n in names]
aucs = [teacher_results[n]['auc'] for n in names]
x = np.arange(len(names)); w = 0.35
b1 = ax.bar(x - w/2, f1s, w, label='F1-Score', color='#2196F3', alpha=0.8)
b2 = ax.bar(x + w/2, aucs, w, label='AUC-ROC', color='#FF9800', alpha=0.8)
for bars in [b1, b2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Score'); ax.set_title('Teacher Model Comparison (10% Labeled Data)\nC1: SSL Strategy | C2: Architecture')
ax.set_xticks(x); ax.set_xticklabels(display, fontsize=9); ax.legend(); ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'teacher_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: teacher_comparison.png")

## 2.9 ML Baselines — XGBoost & Random Forest

Traditional ML models using flattened packet features (32×5 = 160-d vector).
These represent the state-of-the-art **without** deep learning.

In [ ]:
print("Training ML baselines...")
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

X_train = np.stack([d['features'].flatten() for d in train_data])
y_train = np.array([d['label'] for d in train_data])
X_test = np.stack([d['features'].flatten() for d in test_data])
y_test = np.array([d['label'] for d in test_data])

ml_results = {}

for name, clf in [("XGBoost", GradientBoostingClassifier(n_estimators=200, max_depth=6, random_state=42)),
                   ("RandomForest", RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42))]:
    t0 = time.time()
    clf.fit(X_train, y_train)
    train_time = time.time() - t0
    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)[:, 1]
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    ml_results[name] = {'f1': f1, 'auc': auc, 'acc': accuracy_score(y_test, y_pred),
                        'recall': recall_score(y_test, y_pred), 'train_time': train_time}
    print(f"  {name}: F1={f1:.4f} AUC={auc:.4f} ({train_time:.1f}s)")

with open(os.path.join(RESULT_DIR, 'ml_baselines_nb.json'), 'w') as f:
    json.dump(ml_results, f, indent=2)
del X_train, X_test  # free memory

## 2.10 Train 4 Student Variants (Contribution 3 — TED)

All students use the same **BlockwiseStudent** architecture (UniMamba with exits at {8,16,32}).
The only difference is the distillation strategy:
- **No KD**: Supervised CE loss only (no teacher signal)
- **Standard KD**: Distill at pkt 32 only (traditional KD)
- **Uniform KD**: Equal weight at all 3 exits (control)
- **TED (Ours)**: Weight {8:2, 16:1, 32:0.5} — emphasize early exits

In [ ]:
# Load best teacher for distillation
teacher_path = os.path.join(TEACHER_DIR, "teacher_bimamba_masking.pth")
teacher_encoder = BiMambaEncoder(d_model=256)
teacher = Classifier(teacher_encoder, use_proj=True)
teacher.load_state_dict(torch.load(teacher_path, map_location='cpu', weights_only=False))
teacher.eval()
print(f"Teacher loaded: {teacher_path}")

student_configs = OrderedDict({
    "student_no_kd":       ("No KD",       "none"),
    "student_standard_kd": ("Standard KD", "standard"),
    "student_uniform_kd":  ("Uniform KD",  "uniform"),
    "student_ted":         ("TED (Ours)",   "ted"),
})

student_results = {}
for key, (display_name, kd_mode) in student_configs.items():
    save_path = os.path.join(STUDENT_DIR, f"{key}.pth")
    old_path = os.path.join(CKPT_DIR, f"{key}_v2.pth")
    old_path2 = os.path.join(CKPT_DIR, f"{key}.pth")
    if not os.path.exists(save_path):
        if os.path.exists(old_path):
            import shutil; shutil.copy2(old_path, save_path)
        elif os.path.exists(old_path2):
            import shutil; shutil.copy2(old_path2, save_path)

    student = BlockwiseStudent(d_model=256, n_layers=4, exit_points=[8, 16, 32])

    if os.path.exists(save_path):
        print(f"[CACHED] {display_name}")
        student.load_state_dict(torch.load(save_path, map_location='cpu', weights_only=False))
        student.to(DEVICE)
    else:
        print(f"\nTraining: {display_name} (mode={kd_mode})")
        t_use = None if kd_mode == "none" else teacher
        student, _ = train_student(student, t_use, train_data, val_data,
                                    kd_mode=kd_mode, epochs=30, lr=1e-3, batch_size=32,
                                    patience=5, name=display_name, save_path=save_path)

    student.eval()
    exit_metrics = {}
    for ep in [8, 16, 32]:
        ap, al, apr = [], [], []
        with torch.no_grad():
            for x, y in test_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                logits, _ = student.forward_at_exit(x, ep)
                ap.extend(logits.argmax(1).cpu().numpy())
                al.extend(y.cpu().numpy())
                apr.extend(torch.softmax(logits, 1)[:, 1].cpu().numpy())
        exit_metrics[ep] = {
            'f1': f1_score(al, ap, zero_division=0),
            'auc': roc_auc_score(al, apr) if len(set(al)) > 1 else 0.5,
            'recall': recall_score(al, ap, zero_division=0),
            'acc': accuracy_score(al, ap)
        }
    student_results[key] = exit_metrics
    print(f"  {display_name}: F1@8={exit_metrics[8]['f1']:.4f} F1@16={exit_metrics[16]['f1']:.4f} F1@32={exit_metrics[32]['f1']:.4f}")
    student.cpu(); torch.cuda.empty_cache()

with open(os.path.join(RESULT_DIR, 'student_results_nb.json'), 'w') as f:
    json.dump(student_results, f, indent=2)
print("\nStudent results saved.")

## 2.11 Student F1 vs Packets — Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = {'student_no_kd': '#E91E63', 'student_standard_kd': '#9C27B0',
          'student_uniform_kd': '#FF9800', 'student_ted': '#4CAF50'}
labels_map = {'student_no_kd': 'No KD', 'student_standard_kd': 'Standard KD',
              'student_uniform_kd': 'Uniform KD', 'student_ted': 'TED (Ours)'}
for sname, sdata in student_results.items():
    exits = sorted(sdata.keys())
    f1s = [sdata[ep]['f1'] for ep in exits]
    ax.plot(exits, f1s, 'o-', color=colors.get(sname, 'gray'),
            label=labels_map.get(sname, sname), linewidth=2, markersize=8)
    for ep, f1 in zip(exits, f1s):
        ax.annotate(f'{f1:.3f}', (ep, f1), textcoords="offset points",
                    xytext=(0, 10), ha='center', fontsize=8)
ax.set_xlabel('Packets Observed'); ax.set_ylabel('F1-Score')
ax.set_title('F1 vs Packets Observed — TED Closes the Early-Exit Gap')
ax.legend(); ax.set_xticks([8, 16, 32]); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'student_f1_vs_packets.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: student_f1_vs_packets.png")

## 2.12 Dynamic Early Exit — Threshold Sweep (Contribution 4)

Test different confidence thresholds τ ∈ {0.5, 0.7, 0.8, 0.85, 0.9, 0.95, 0.99}.
At each threshold, measure: F1, average packets used, exit distribution, throughput.

In [ ]:
thresholds = [0.5, 0.7, 0.8, 0.85, 0.9, 0.95, 0.99]
all_exit_results = {}

for sname in student_configs.keys():
    ckpt = os.path.join(STUDENT_DIR, f"{sname}.pth")
    if not os.path.exists(ckpt): continue
    student = BlockwiseStudent(d_model=256, n_layers=4, exit_points=[8, 16, 32])
    student.load_state_dict(torch.load(ckpt, map_location='cpu', weights_only=False))
    student.to(DEVICE).eval()

    thr_results = {}
    for thr in thresholds:
        ap, al, ae = [], [], []
        t0 = time.time()
        with torch.no_grad():
            for x, y in test_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                logits, exits = student.forward_dynamic(x, threshold=thr)
                ap.extend(logits.argmax(1).cpu().numpy())
                al.extend(y.cpu().numpy())
                ae.extend(exits.cpu().numpy())
        elapsed = time.time() - t0
        ec = Counter(ae)
        thr_results[str(thr)] = {
            'f1': f1_score(al, ap, zero_division=0),
            'recall': recall_score(al, ap, zero_division=0),
            'avg_pkts': np.mean(ae),
            'pct_8': ec.get(8,0)/len(ae)*100,
            'pct_16': ec.get(16,0)/len(ae)*100,
            'pct_32': ec.get(32,0)/len(ae)*100,
            'throughput': len(al) / elapsed
        }
    all_exit_results[sname] = thr_results
    print(f"{sname} @ τ=0.9: F1={thr_results['0.9']['f1']:.4f} AvgPkt={thr_results['0.9']['avg_pkts']:.1f} @8={thr_results['0.9']['pct_8']:.0f}%")
    student.cpu(); torch.cuda.empty_cache()

with open(os.path.join(RESULT_DIR, 'early_exit_nb.json'), 'w') as f:
    json.dump(all_exit_results, f, indent=2)

## 2.13 Exit Distribution Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for idx, (sname, thrs) in enumerate(all_exit_results.items()):
    if idx >= 4: break
    ax = axes[idx // 2][idx % 2]
    r = thrs.get('0.9', list(thrs.values())[0])
    vals = [r['pct_8'], r['pct_16'], r['pct_32']]
    bars = ax.bar(['8 pkts', '16 pkts', '32 pkts'], vals,
                  color=['#4CAF50', '#FF9800', '#2196F3'], alpha=0.8)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2., bar.get_height()+1, f'{v:.1f}%', ha='center')
    ax.set_ylabel('% of Flows')
    nm = labels_map.get(sname, sname)
    ax.set_title(f'{nm}\nτ=0.9, Avg={r["avg_pkts"]:.1f} pkts, F1={r["f1"]:.3f}')
    ax.set_ylim(0, 100); ax.grid(axis='y', alpha=0.3)
plt.suptitle('Dynamic Early Exit Distribution (τ=0.9)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'exit_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

## 2.14 Cross-Dataset Evaluation (UNSW → CTU-13)

Test all teachers and students on the CTU-13 botnet dataset.
This evaluates **generalization** — whether SSL representations transfer to unseen traffic.

**Why cross-dataset F1 isn't 99%:** The model was trained on UNSW-NB15 (Australian traffic,
2015). CTU-13 has completely different traffic patterns (Czech university, botnet-specific).
Feature distributions shift: different protocol mixes, different packet sizes, different IATs.
This is the fundamental challenge of zero-shot network traffic generalization.

In [ ]:
cross_results = {}

if os.path.exists(CTU_PKL):
    with open(CTU_PKL, 'rb') as f:
        ctu_data = pickle.load(f)
    cl = Counter(d['label'] for d in ctu_data)
    print(f"CTU-13: {len(ctu_data):,} flows (B={cl[0]:,}, A={cl[1]:,})")
    ctu_a = [d for d in ctu_data if d['label'] == 1][:10000]
    ctu_b = [d for d in ctu_data if d['label'] == 0][:10000]
    cross_eval = ctu_b + ctu_a
    cross_loader = DataLoader(FlowDataset(cross_eval), batch_size=64, num_workers=2)

    # Test all teachers
    for key in teacher_configs.keys():
        ckpt = os.path.join(TEACHER_DIR, f"teacher_{key}.pth")
        if not os.path.exists(ckpt): continue
        enc_cls = BiMambaEncoder if 'bimamba' in key else BertEncoder
        enc = enc_cls(d_model=256)
        model = Classifier(enc, use_proj=True)
        model.load_state_dict(torch.load(ckpt, map_location='cpu', weights_only=False))
        model.to(DEVICE)
        m = evaluate_model(model, cross_loader)
        cross_results[f"teacher_{key}"] = m
        print(f"  teacher_{key}: F1={m['f1']:.4f} AUC={m['auc']:.4f}")
        model.cpu(); torch.cuda.empty_cache()

    # Test TED student
    for sname in ['student_ted', 'student_uniform_kd']:
        ckpt = os.path.join(STUDENT_DIR, f"{sname}.pth")
        if not os.path.exists(ckpt): continue
        student = BlockwiseStudent(d_model=256, n_layers=4, exit_points=[8, 16, 32])
        student.load_state_dict(torch.load(ckpt, map_location='cpu', weights_only=False))
        student.to(DEVICE).eval()
        ap, al, apr = [], [], []
        with torch.no_grad():
            for x, y in cross_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                logits, _ = student.forward_at_exit(x, 32)
                ap.extend(logits.argmax(1).cpu().numpy())
                al.extend(y.cpu().numpy())
                apr.extend(torch.softmax(logits, 1)[:,1].cpu().numpy())
        f1 = f1_score(al, ap, zero_division=0)
        auc = roc_auc_score(al, apr) if len(set(al)) > 1 else 0.5
        cross_results[sname] = {'f1': f1, 'auc': auc}
        print(f"  {sname}: F1={f1:.4f} AUC={auc:.4f}")
        student.cpu(); torch.cuda.empty_cache()
    del ctu_data
else:
    print("CTU-13 data not found — skipping cross-dataset eval.")

with open(os.path.join(RESULT_DIR, 'cross_dataset_nb.json'), 'w') as f:
    json.dump(cross_results, f, indent=2)

## 2.15 Cross-Dataset Visualization

In [ ]:
if cross_results:
    fig, ax = plt.subplots(figsize=(12, 6))
    names = [k for k in cross_results.keys() if 'f1' in cross_results[k]]
    f1s = [cross_results[n]['f1'] for n in names]
    display = [n.replace('teacher_', '').replace('student_', 'S:').replace('_', '\n') for n in names]
    bars = ax.bar(range(len(names)), f1s, color=['#4CAF50' if f1 > 0.3 else '#E91E63' for f1 in f1s], alpha=0.8)
    for bar, f1 in zip(bars, f1s):
        ax.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.01, f'{f1:.3f}', ha='center', fontsize=9)
    ax.set_xticks(range(len(names))); ax.set_xticklabels(display, fontsize=8)
    ax.set_ylabel('F1-Score'); ax.set_title('Cross-Dataset Generalization: UNSW-NB15 → CTU-13')
    ax.grid(axis='y', alpha=0.3); plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, 'cross_dataset.png'), dpi=150, bbox_inches='tight')
    plt.show()

## 2.16 Per-Attack F1 Breakdown — Binary vs Macro

UNSW-NB15 is ~97% benign. **Binary F1** (attack vs benign) is dominated by the benign
majority — a model that mostly gets benign right scores high.

**Macro F1** (unweighted average across per-category F1) reveals true detection capability.
This is critical for NIDS: missing rare attacks (Worms, Analysis) is dangerous.

In [ ]:
print("Computing per-attack breakdown...")
# Load all flows for comprehensive eval
with open(FLOWS_ALL_PKL, 'rb') as f:
    flows_all = pickle.load(f)

# Split for per-category eval (use 30% test)
all_labels = np.array([d['label'] for d in flows_all])
_, test_idx_all = train_test_split(np.arange(len(flows_all)), test_size=0.3,
                                    stratify=all_labels, random_state=42)
eval_flows = [flows_all[i] for i in test_idx_all]
eval_loader_all = DataLoader(FlowDataset(eval_flows), batch_size=128, num_workers=2)

# Get attack categories
categories = sorted(set(d['label_str'] for d in eval_flows if d['label'] == 1))
print(f"Attack categories: {categories}")

def per_category_metrics(model, loader, eval_data, model_name):
    model.eval()
    ap, al = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            logits = model(x)
            ap.extend(logits.argmax(1).cpu().numpy())
            al.extend(y.cpu().numpy())
    ap, al = np.array(ap), np.array(al)
    binary_f1 = f1_score(al, ap, zero_division=0)

    # Per-category recall
    cat_f1s = []
    for cat in categories:
        cat_idx = [i for i, d in enumerate(eval_data) if d['label_str'] == cat]
        if not cat_idx: continue
        cat_labels = al[cat_idx]
        cat_preds = ap[cat_idx]
        rec = recall_score(cat_labels, cat_preds, zero_division=0)
        cat_f1s.append(rec)

    macro_f1 = np.mean(cat_f1s) if cat_f1s else 0
    return binary_f1, macro_f1, cat_f1s

# Evaluate BiMamba teacher
ckpt = os.path.join(TEACHER_DIR, "teacher_bimamba_masking.pth")
enc = BiMambaEncoder(d_model=256)
model = Classifier(enc, use_proj=True)
model.load_state_dict(torch.load(ckpt, map_location='cpu', weights_only=False))
model.to(DEVICE)
bm_bin, bm_mac, bm_cats = per_category_metrics(model, eval_loader_all, eval_flows, "BiMamba")
model.cpu()

print(f"\nBiMamba+Masking:")
print(f"  Binary F1: {bm_bin:.4f}")
print(f"  Macro F1:  {bm_mac:.4f}")
for cat, rec in zip(categories, bm_cats):
    print(f"    {cat:20s}: {rec:.3f}")

# Horizontal bar chart
fig, ax = plt.subplots(figsize=(10, 6))
y_pos = np.arange(len(categories))
ax.barh(y_pos, bm_cats, color='#4CAF50', alpha=0.8)
ax.set_yticks(y_pos); ax.set_yticklabels(categories)
ax.set_xlabel('Recall'); ax.set_title('Per-Attack Recall — BiMamba+Masking SSL')
ax.set_xlim(0, 1.05)
for i, v in enumerate(bm_cats):
    ax.text(v + 0.01, i, f'{v:.1%}', va='center')
ax.grid(axis='x', alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'per_attack_recall.png'), dpi=150, bbox_inches='tight')
plt.show()
del flows_all
torch.cuda.empty_cache()

## 2.17 Time-to-Detection (TTD) from Raw Packet Data

TTD is computed directly from **raw inter-arrival times (IATs)** in the preprocessed flows.

TTD at k packets = sum of the first k−1 IATs per flow.

This is the actual time a NIDS would wait before having enough packets to classify.
We compute TTD at exit points {8, 16, 32} and the TED-weighted average.

In [ ]:
print("Computing TTD from raw packet data...")
with open(FLOWS_ALL_PKL, 'rb') as f:
    flows_ttd = pickle.load(f)

attack_flows = [d for d in flows_ttd if d['label'] == 1]
benign_flows = [d for d in flows_ttd if d['label'] == 0]
print(f"Total flows: {len(flows_ttd):,} (attack: {len(attack_flows):,}, benign: {len(benign_flows):,})")

def compute_ttd(flows, k):
    """TTD at k packets = sum of first k-1 IATs."""
    ttds = []
    for flow in flows:
        iats = flow['features'][:k, 3]  # Column 3 = IAT
        ttd = np.sum(iats[:k-1]) * 1000  # Convert to ms
        ttds.append(ttd)
    return np.array(ttds)

# TTD for attack flows at each exit point
print("\n=== TTD (Attack Flows Only) ===")
ttd_table = {}
for k in [8, 16, 32]:
    ttds = compute_ttd(attack_flows, k)
    ttd_table[k] = {
        'mean': float(np.mean(ttds)),
        'median': float(np.median(ttds)),
        'p90': float(np.percentile(ttds, 90))
    }
    print(f"  TTD@{k:2d} pkts: mean={np.mean(ttds):.0f}ms  median={np.median(ttds):.0f}ms  P90={np.percentile(ttds, 90):.0f}ms")

# TED-weighted TTD: 66.8% exit@8, 16.1% exit@16, 17.1% exit@32
ttd8 = compute_ttd(attack_flows, 8)
ttd16 = compute_ttd(attack_flows, 16)
ttd32 = compute_ttd(attack_flows, 32)
ted_ttd = 0.668 * ttd8 + 0.161 * ttd16 + 0.171 * ttd32
print(f"\n  TED weighted: mean={np.mean(ted_ttd):.0f}ms  median={np.median(ted_ttd):.0f}ms")
print(f"  Speedup: {np.mean(ttd32)/np.mean(ted_ttd):.1f}x")
ttd_table['ted_weighted'] = {
    'mean': float(np.mean(ted_ttd)),
    'median': float(np.median(ted_ttd)),
    'speedup': float(np.mean(ttd32) / np.mean(ted_ttd))
}

# Mean IAT statistics
all_iats = np.concatenate([f['features'][:, 3] for f in attack_flows])
print(f"\n  Mean attack IAT: {np.mean(all_iats)*1000:.1f}ms  Median: {np.median(all_iats)*1000:.2f}ms")

## 2.18 TTD by Attack Category

In [ ]:
# Per-attack category TTD
cat_ttd = {}
for cat in set(d['label_str'] for d in attack_flows):
    cat_flows = [d for d in attack_flows if d['label_str'] == cat]
    if len(cat_flows) < 10: continue
    t8 = compute_ttd(cat_flows, 8)
    t32 = compute_ttd(cat_flows, 32)
    cat_ttd[cat] = {
        'n': len(cat_flows),
        'ttd_8_mean': float(np.mean(t8)),
        'ttd_32_mean': float(np.mean(t32)),
        'speedup': float(np.mean(t32) / np.mean(t8)) if np.mean(t8) > 0 else 0
    }

print("\n=== TTD by Attack Category ===")
print(f"{'Category':20s} {'N':>6s} {'TTD-8':>10s} {'TTD-32':>10s} {'Speedup':>8s}")
print("-" * 58)
for cat, data in sorted(cat_ttd.items(), key=lambda x: x[1]['ttd_8_mean']):
    print(f"{cat:20s} {data['n']:>6,} {data['ttd_8_mean']:>8.0f}ms {data['ttd_32_mean']:>8.0f}ms {data['speedup']:>6.1f}x")

# Save TTD results
ttd_results = {'overall': ttd_table, 'per_category': cat_ttd}
with open(os.path.join(RESULT_DIR, 'ttd_results.json'), 'w') as f:
    json.dump(ttd_results, f, indent=2)

# Visualization
cats = sorted(cat_ttd.keys(), key=lambda c: cat_ttd[c]['ttd_8_mean'])
fig, ax = plt.subplots(figsize=(12, 6))
y_pos = np.arange(len(cats))
t8_vals = [cat_ttd[c]['ttd_8_mean'] for c in cats]
t32_vals = [cat_ttd[c]['ttd_32_mean'] for c in cats]
h = 0.35
ax.barh(y_pos - h/2, t32_vals, h, label='TTD-32 (full buffer)', color='#90CAF9', alpha=0.8)
ax.barh(y_pos + h/2, t8_vals, h, label='TTD-8 (early exit)', color='#4CAF50', alpha=0.8)
ax.set_yticks(y_pos); ax.set_yticklabels(cats)
ax.set_xlabel('Time-to-Detection (ms)'); ax.set_title('TTD by Attack Category — From Raw Packet IATs')
ax.legend(); ax.grid(axis='x', alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'ttd_by_category.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ttd_by_category.png")
del flows_ttd, attack_flows, benign_flows

## 2.19 Latency & Throughput Benchmarking

Measure actual inference latency for each model at different batch sizes.
This quantifies the **computational cost** story:
- BERT: O(n²) attention → slower per-sample at large sequences
- BiMamba: O(n) scan but sequential recurrence → moderate
- UniMamba student: O(n) forward-only → fastest

In [ ]:
def benchmark_latency(model, input_fn, batch_sizes=[1, 64, 256], warmup=10, n_runs=50):
    model.to(DEVICE).eval()
    results = {}
    for bs in batch_sizes:
        x = input_fn(bs).to(DEVICE)
        # Warmup
        with torch.no_grad():
            for _ in range(warmup):
                _ = model(x)
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        # Benchmark
        times = []
        for _ in range(n_runs):
            if torch.cuda.is_available(): torch.cuda.synchronize()
            t0 = time.perf_counter()
            with torch.no_grad():
                _ = model(x)
            if torch.cuda.is_available(): torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1000)
        mean_ms = np.mean(times)
        results[bs] = {
            'total_ms': float(mean_ms),
            'per_sample_ms': float(mean_ms / bs),
            'throughput': float(bs / (mean_ms / 1000))
        }
    model.cpu()
    return results

input_fn = lambda bs: torch.randn(bs, 32, 5)
latency_results = {}

# BERT teacher
enc = BertEncoder(256)
ckpt = os.path.join(TEACHER_DIR, "teacher_bert_masking.pth")
if os.path.exists(ckpt):
    model = Classifier(enc, use_proj=True)
    model.load_state_dict(torch.load(ckpt, map_location='cpu', weights_only=False))
else:
    model = Classifier(enc, use_proj=True)
latency_results['BERT'] = benchmark_latency(model, input_fn)
print(f"BERT: {latency_results['BERT'][256]['throughput']:.0f} fl/s @ bs=256")

# BiMamba teacher
enc = BiMambaEncoder(256)
model = Classifier(enc, use_proj=True)
ckpt = os.path.join(TEACHER_DIR, "teacher_bimamba_masking.pth")
if os.path.exists(ckpt):
    model.load_state_dict(torch.load(ckpt, map_location='cpu', weights_only=False))
latency_results['BiMamba'] = benchmark_latency(model, input_fn)
print(f"BiMamba: {latency_results['BiMamba'][256]['throughput']:.0f} fl/s @ bs=256")

# TED Student @8
student = BlockwiseStudent(256, 4, [8, 16, 32])
ckpt = os.path.join(STUDENT_DIR, "student_ted.pth")
if os.path.exists(ckpt):
    student.load_state_dict(torch.load(ckpt, map_location='cpu', weights_only=False))

class StudentAt8Wrapper(nn.Module):
    def __init__(self, s): super().__init__(); self.s = s
    def forward(self, x):
        logits, _ = self.s.forward_at_exit(x, 8)
        return logits
wrapper = StudentAt8Wrapper(student)
latency_results['TED@8'] = benchmark_latency(wrapper, input_fn)
print(f"TED@8: {latency_results['TED@8'][256]['throughput']:.0f} fl/s @ bs=256")

with open(os.path.join(RESULT_DIR, 'latency_nb.json'), 'w') as f:
    json.dump(latency_results, f, indent=2)
torch.cuda.empty_cache()

## 2.20 Model Size Comparison

In [ ]:
print(f"{'Model':<30s} {'Params':>12s} {'Weight File':>12s}")
print("-" * 58)
model_sizes = {}
for name, cls in [("BERT Teacher", BertEncoder), ("BiMamba Teacher", BiMambaEncoder)]:
    enc = cls(256)
    m = Classifier(enc, use_proj=True)
    params = sum(p.numel() for p in m.parameters())
    model_sizes[name] = params
    print(f"{name:<30s} {params:>12,}")

s = BlockwiseStudent(256, 4, [8, 16, 32])
params = sum(p.numel() for p in s.parameters())
model_sizes["TED Student"] = params
print(f"{'TED Student':<30s} {params:>12,}")

# Weight file sizes
print("\nSaved weight files:")
for d in [SSL_DIR, TEACHER_DIR, STUDENT_DIR]:
    for fn in sorted(os.listdir(d)):
        fp = os.path.join(d, fn)
        print(f"  {fn}: {os.path.getsize(fp)/1e6:.1f} MB")

## 2.21 IAT Distribution Histogram

Inter-arrival times determine Time-to-Detection. This histogram shows the bimodal
distribution: burst packets (near-simultaneous, IAT < 1ms) vs long gaps (seconds).

In [ ]:
# Sample attack flows for IAT histogram
with open(FLOWS_ALL_PKL, 'rb') as f:
    flows_hist = pickle.load(f)
attack_hist = [d for d in flows_hist if d['label'] == 1]

iats = np.concatenate([f['features'][:, 3] for f in attack_hist[:10000]])
iats_nonzero = iats[iats > 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
axes[0].hist(iats_nonzero * 1000, bins=100, alpha=0.7, color='steelblue', edgecolor='white')
axes[0].set_xlabel('IAT (ms)'); axes[0].set_ylabel('Count')
axes[0].set_title('Attack Packet IAT Distribution (Linear)')
axes[0].axvline(np.mean(iats_nonzero) * 1000, color='red', linestyle='--', label=f'Mean={np.mean(iats_nonzero)*1000:.1f}ms')
axes[0].axvline(np.median(iats_nonzero) * 1000, color='orange', linestyle='--', label=f'Median={np.median(iats_nonzero)*1000:.2f}ms')
axes[0].legend(); axes[0].set_xlim(0, 200)

# Log scale
log_iats = np.log10(iats_nonzero * 1000 + 0.001)
axes[1].hist(log_iats, bins=100, alpha=0.7, color='#4CAF50', edgecolor='white')
axes[1].set_xlabel('log₁₀(IAT in ms)'); axes[1].set_ylabel('Count')
axes[1].set_title('Attack Packet IAT Distribution (Log Scale)')

plt.suptitle('Inter-Arrival Time Distribution — Attack Packets', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'iat_histogram.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: iat_histogram.png")
del flows_hist, attack_hist

## 2.22 Speed-Accuracy Pareto Front

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
# Plot teachers
for key, m in teacher_results.items():
    if m['f1'] > 0.01:
        ttd = ttd_table[32]['mean'] if '32' in str(ttd_table) else 804
        marker = 's' if 'bert' in key else 'o'
        ax.scatter(ttd, m['f1'], s=120, marker=marker, zorder=5,
                   label=f"{key} (F1={m['f1']:.3f})")

# Plot ML
for name, m in ml_results.items():
    ax.scatter(0, m['f1'], s=100, marker='D', zorder=5,
               label=f"{name} (F1={m['f1']:.3f}, instant)")

# Plot TED student (dynamic)
if 'student_ted' in all_exit_results:
    r = all_exit_results['student_ted'].get('0.9', {})
    if r:
        ted_mean_ttd = ttd_table.get('ted_weighted', {}).get('mean', 427)
        ax.scatter(ted_mean_ttd, r['f1'], s=200, marker='*', c='red', zorder=10,
                   label=f"TED Student (F1={r['f1']:.3f}, TTD={ted_mean_ttd:.0f}ms)")

ax.set_xlabel('Time-to-Detection (ms)', fontsize=12)
ax.set_ylabel('F1-Score', fontsize=12)
ax.set_title('Speed-Accuracy Pareto Front', fontsize=13)
ax.legend(loc='lower right', fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'pareto_front.png'), dpi=150, bbox_inches='tight')
plt.show()

## 2.23 Final Summary — All Results

Comprehensive comparison across all models and metrics.

In [ ]:
print("=" * 90)
print("  COMPREHENSIVE RESULTS SUMMARY")
print("=" * 90)

print(f"\n{'Model':<30s} {'F1':>8s} {'AUC':>8s} {'Recall':>8s} {'Cross F1':>10s}")
print("-" * 70)

# Teachers
for key, m in teacher_results.items():
    cf1 = cross_results.get(f"teacher_{key}", {}).get('f1', '-')
    cf1_str = f"{cf1:.4f}" if isinstance(cf1, float) else cf1
    print(f"T: {key:<27s} {m['f1']:>8.4f} {m['auc']:>8.4f} {m['recall']:>8.4f} {cf1_str:>10s}")

# ML
for name, m in ml_results.items():
    print(f"ML: {name:<26s} {m['f1']:>8.4f} {m['auc']:>8.4f} {m['recall']:>8.4f} {'N/A':>10s}")

# Students
for key, m in student_results.items():
    cf1 = cross_results.get(key, {}).get('f1', '-')
    cf1_str = f"{cf1:.4f}" if isinstance(cf1, float) else cf1
    print(f"S: {key:<27s} {m[32]['f1']:>8.4f} {m[32]['auc']:>8.4f} {m[32]['recall']:>8.4f} {cf1_str:>10s}")

# TTD
print(f"\n{'TTD Summary':>30s}")
print(f"  TTD@32 (full buffer): {ttd_table[32]['mean']:.0f}ms")
print(f"  TTD@8 (early exit):   {ttd_table[8]['mean']:.0f}ms")
print(f"  TED weighted:         {ttd_table['ted_weighted']['mean']:.0f}ms")
print(f"  Speedup:              {ttd_table['ted_weighted']['speedup']:.1f}x")

print(f"\n  All results saved to: {RESULT_DIR}")
print(f"  All plots saved to:   {PLOT_DIR}")
print("=" * 90)
print("  PART 2 COMPLETE — ALL EVALUATION DONE")
print("=" * 90)